# Welcome to NER Finetuning with HF Transformers 

First of all, we need to import all necessary libraries and modules.

## Contents 
1. Configurations (Resources, Training, Optimization, Evaluation, Saving results)
2. Prepare training & dataset
3. Train (Finetune) a NER Model
4. Optimization
5. Evaluation

In [1]:
# import modules
import config
import train
import eval_opt

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configurations

In this section, we set all of the configurations for the following steps (training, optimization, evaluation). Depending on which parts you want to run and which to leave out, corresponding config parts can be skipped if not needed.

### Resources

We need to state which resources we want to use for NER finetuning, namely the `dataset` and `model`. Therefore, we use the `Resource` class and define the `path` (either to a HF or a local directory) and `source` (`hf` or `local`) of our resources. By applying `set_name()`, the model or dataset name gets set automatically, otherwise the `model.name` can also be updated manually.

In [2]:
"""
#uncomment and delete other section later for explaination purposes
model = config.Resource(path="test/test", source="hf")
model.name = "my_model_name"

dataset = config.Resource(path="dir/dataset", source="local")
dataset.set_name()
"""

model = config.Resource(path="FacebookAI/xlm-roberta-base", source="hf")
model.set_name()

dataset = config.Resource(path="data/zefys2025.hf", source="local")
dataset.set_name()

In [3]:
#check our model and dataset configurations again
print(model.info())
print(dataset.info())

xlm-roberta-base will be loaded from FacebookAI/xlm-roberta-base (via hf).
zefys2025.hf will be loaded from data/zefys2025.hf (via local).


### Training

You can define a variable for the training parameters based on our default `TrainingParams` config and try to improve results using evaluation/optimization techniques, or start by overwriting the parameters for training manually. Training will only be performed once on the training dataset and evaluated once on the validation dataset. For more sophisticated methods, you can additionally run the Optimization and Evaluation parts to find ideal hyperparameter combinations and perform a more reliable evaluation.

In [4]:
training_params = config.TrainingParams()
training_params.batch_size = 32
training_params.__dict__

{'batch_size': 32,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01,
 'warmup_steps': 100}

### Optimization

For optimization (hyperparameter search) we use the HF Trainer API (https://huggingface.co/docs/transformers/hpo_train#hyperparameter-search-using-trainer-api) and optuna (https://optuna.readthedocs.io/en/stable/index.html, `pip install optuna`). We can access all of the search parameters, as they are stored in a dict structure, and also adapt the number of trials `n_trials` to run. 

In [5]:
optimize_params = config.OptimizeParams()

optimize_params.hp_space["learning_rate"]["param_type"]
optimize_params.__dict__

{'hp_space': {'learning_rate': {'param_type': 'float',
   'borders': [1e-06, 0.0001]},
  'warmup_steps': {'param_type': 'categorical', 'borders': [20, 40, 160]}},
 'n_trials': 20}

In [6]:
#to remove an hyperparameter from the hp_space dict, use pop() or similar
#optimize_params.hp_space.pop("learning_rate")
optimize_params.__dict__

{'hp_space': {'learning_rate': {'param_type': 'float',
   'borders': [1e-06, 0.0001]},
  'warmup_steps': {'param_type': 'categorical', 'borders': [20, 40, 160]}},
 'n_trials': 20}

### Evaluation and optimization

* Evaluation: Stratified k-fold Cross-Validation - one of ["None", k] (https://huggingface.co/docs/datasets/v1.11.0/splits.html#examples; https://discuss.huggingface.co/t/k-fold-cross-validation/5765/5, https://github.com/huggingface/datasets/discussions/5940)
* Optimization:
  * https://huggingface.co/docs/transformers/hpo_train
  * https://medium.com/carbon-consulting/transformer-models-hyperparameter-optimization-with-the-optuna-299e185044a8
  * Grid Search (GridSearchCV?), Line Search, hyperopt
* Analyze grid search results using Heatmap or similar

### Saving results
* save configuration
* create/save plots

## 2. Prepare training & dataset

In [7]:
train.set_torch_device()

device(type='cuda')

In [8]:
ner_dataset = train.load_ner_dataset(dataset.path, dataset.source)
#ner_dataset["train"][0]

In [9]:
tokenizer = train.get_tokenizer(model.path)

In [10]:
tokenized_dataset = train.prepare_dataset(ner_dataset, tokenizer)
label_list = train.get_label_list(ner_dataset)

In [11]:
label_list

['I-PER', 'B-PER', 'O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG']

## 3. Train (Finetune) a NER Model

To finetune our model for NER, we first need to define our `ner` task (see train.py). 

In [ ]:
trained_ner_model = train.train_model(model.path, label_list, training_params, tokenized_dataset, tokenizer)

## 4. Optimization

In [12]:
training_params.batch_size = 64
training_params.num_train_epochs = 1
optimize_params.n_trials = 10

In [13]:
eval_opt.optimize(optimize_params, training_params, model.path, label_list, tokenizer, tokenized_dataset)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-03-31 15:33:51,853] A new study created in memory with name: no-name-a1c50b1d-da48-491e-ac33-41056ede890d
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


optimizing within the following hyperparameter space:
{'learning_rate': 2.832650801625541e-06, 'warmup_steps': 40}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.212664,0.000000,0.000000,0.000000,0.921955


[I 2025-03-31 15:37:55,317] Trial 0 finished with value: 0.9219549046954905 and parameters: {'learning_rate': 2.832650801625541e-06, 'warmup_steps': 40}. Best is trial 0 with value: 0.9219549046954905.


optimizing within the following hyperparameter space:
{'learning_rate': 5.530369723962656e-05, 'warmup_steps': 40}


...

optimizing within the following hyperparameter space:
{'learning_rate': 1.3017854571235095e-06, 'warmup_steps': 20}


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.388946,0.000000,0.000000,0.000000,0.921606


[I 2025-03-31 16:14:15,197] Trial 9 pruned. 


BestRun(run_id='1', objective=4.0, hyperparameters={'learning_rate': 5.530369723962656e-05, 'warmup_steps': 40}, run_summary=None)

## 5. Evaluation

In [ ]:
eval_opt.compute_metrics_per_tag(trained_ner_model, tokenized_dataset, label_list) 